In [8]:
"""
ETH Whale Activity ML Pipeline - With Step 1 Target Audit
Complete Daily Pipeline with Formal Directional Target Verification
"""

import os
import time
import json
import requests
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score
from dotenv import load_dotenv
import joblib
import warnings
warnings.filterwarnings('ignore')

# ========== CONFIGURATION ==========
load_dotenv()
DUNE_API_KEY = os.getenv("DUNE_WHALES_API")
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")
os.makedirs("data", exist_ok=True)
os.makedirs("data/price_cache", exist_ok=True)

QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

PRICE = [
    'eth_ret_lag1','eth_ret_lag3','eth_ret_lag7',
    'eth_vol7','eth_vol30','eth_rsi',
    'btc_ret_lag1','btc_ret_lag3','btc_ret_lag7',
    'btc_vol7','btc_vol30','btc_rsi',
    'eth_btc_ratio','eth_btc_ratio_ma7','eth_btc_corr_30d'
]

ONCHAIN = [
    'whale_tx_zscore_90d','whale_volume_ratio',
    'whale_volume_ratio_delta_1d','whale_volume_ratio_delta_3d',
    'exchange_flow_share','net_exchange_flow_ratio',
    'whale_exchange_flow_ratio','whale_exchange_asymmetry',
    'tx_per_active_zscore_90d','eth_burned_zscore_90d'
]

HYBRID = PRICE + ONCHAIN

# ========== STEP 1: TARGET AUDIT FUNCTIONS ==========

def audit_target_structure(df, target_col='direction_t2', n_splits=5):
    """
    STEP 1 — FORMAL DIRECTIONAL TARGET AUDIT
    Verifies target is learnable, balanced, and correctly evaluated
    """
    print("\n" + "="*70)
    print("STEP 1 — FORMAL DIRECTIONAL TARGET AUDIT")
    print("="*70)
    
    # Prepare data
    target_cols = ['direction_t2', 'direction_t3', 'return_t2', 'return_t3', 'block_date']
    X = df.drop(columns=target_cols, errors='ignore').dropna()
    y = df.loc[X.index, target_col]
    
    # Also get returns for magnitude analysis
    returns = df.loc[X.index, 'return_t2'] if 'return_t2' in df.columns else None
    
    audit_results = {}
    
    # 1️⃣ GLOBAL CLASS BALANCE
    print("\n1️⃣ GLOBAL CLASS BALANCE")
    print("-" * 70)
    
    class_dist = y.value_counts(normalize=True).sort_index()
    up_pct = class_dist.get(1, 0) * 100
    down_pct = class_dist.get(0, 0) * 100
    
    print(f"DOWN (0): {down_pct:.1f}%")
    print(f"UP   (1): {up_pct:.1f}%")
    
    # Assess balance
    balance_ratio = min(up_pct, down_pct) / max(up_pct, down_pct)
    if balance_ratio >= 0.80:
        status = "✅ EXCELLENT"
    elif balance_ratio >= 0.67:
        status = "⚠️  MANAGEABLE"
    elif balance_ratio >= 0.54:
        status = "⚠️  BIAS RISK"
    else:
        status = "🚨 CRITICAL - Model will collapse to majority"
    
    print(f"\nBalance Status: {status}")
    audit_results['global_balance'] = {'up_pct': up_pct, 'down_pct': down_pct, 'status': status}
    
    # 2️⃣ PER-FOLD CLASS BALANCE
    print("\n2️⃣ PER-FOLD CLASS BALANCE (CRITICAL)")
    print("-" * 70)
    
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=30)
    fold_issues = []
    
    for fold, (_, test_idx) in enumerate(tscv.split(X), 1):
        y_fold = y.iloc[test_idx]
        counts = y_fold.value_counts().sort_index()
        
        down_count = counts.get(0, 0)
        up_count = counts.get(1, 0)
        
        # Check for critical issues
        issue = None
        if down_count < 10:
            issue = f"🚨 <10 DOWN samples ({down_count})"
        elif up_count < 10:
            issue = f"🚨 <10 UP samples ({up_count})"
        elif down_count == 0 or up_count == 0:
            issue = "🚨 SINGLE CLASS FOLD"
        
        status_icon = "🚨" if issue else "✅"
        print(f"Fold {fold}: DOWN={down_count:2d}, UP={up_count:2d} {status_icon}")
        
        if issue:
            fold_issues.append((fold, issue))
            print(f"         {issue}")
    
    if fold_issues:
        print(f"\n⚠️  {len(fold_issues)} folds have critical issues")
        print("   This explains P↓=0.000 or P↓=1.000 instability")
    else:
        print("\n✅ All folds have adequate samples")
    
    audit_results['fold_balance'] = {'issues': fold_issues, 'n_folds': n_splits}
    
    # 3️⃣ BASE-RATE BENCHMARK
    print("\n3️⃣ BASE-RATE BENCHMARK (Reality Check)")
    print("-" * 70)
    
    baseline_acc = max(y.mean(), 1 - y.mean())
    print(f"Baseline Accuracy (majority class): {baseline_acc:.4f}")
    print(f"\n⚠️  Any model scoring < {baseline_acc:.4f} is statistically useless")
    
    audit_results['baseline'] = baseline_acc
    
    # 4️⃣ MOVE MAGNITUDE AUDIT
    if returns is not None:
        print("\n4️⃣ MOVE MAGNITUDE AUDIT (Noise Detection)")
        print("-" * 70)
        
        abs_returns = returns.abs()
        print("\nReturn Magnitude Distribution:")
        print(abs_returns.describe().to_string())
        
        # Check for coin-flip territory
        noise_threshold = 0.002
        noise_pct = (abs_returns < noise_threshold).mean() * 100
        
        print(f"\nMoves < {noise_threshold:.1%}: {noise_pct:.1f}%")
        
        if noise_pct > 50:
            print("🚨 CRITICAL: >50% moves are noise - predicting coin flips")
        elif noise_pct > 40:
            print("⚠️  WARNING: High noise ratio - consider volatility gating")
        else:
            print("✅ Acceptable signal-to-noise ratio")
        
        audit_results['noise'] = {'pct_small_moves': noise_pct, 'threshold': noise_threshold}
    
    # 5️⃣ LABEL STABILITY TEST
    if all(f'direction_t{h}' in df.columns for h in [1, 2, 3]):
        print("\n5️⃣ LABEL STABILITY TEST (T+1 vs T+2 vs T+3)")
        print("-" * 70)
        
        valid_idx = df[['direction_t1', 'direction_t2', 'direction_t3']].dropna().index
        
        align_12 = (df.loc[valid_idx, 'direction_t1'] == df.loc[valid_idx, 'direction_t2']).mean()
        align_13 = (df.loc[valid_idx, 'direction_t1'] == df.loc[valid_idx, 'direction_t3']).mean()
        align_23 = (df.loc[valid_idx, 'direction_t2'] == df.loc[valid_idx, 'direction_t3']).mean()
        
        print(f"T+1 vs T+2 agreement: {align_12:.1%}")
        print(f"T+1 vs T+3 agreement: {align_13:.1%}")
        print(f"T+2 vs T+3 agreement: {align_23:.1%}")
        
        if min(align_12, align_13, align_23) < 0.60:
            print("\n⚠️  WARNING: Low directional stability (<60%)")
            print("   This explains why T+2 may outperform T+1")
        else:
            print("\n✅ Acceptable directional stability")
        
        audit_results['stability'] = {
            't1_t2': align_12,
            't1_t3': align_13,
            't2_t3': align_23
        }
    
    # ✅ FINAL VERDICT
    print("\n" + "="*70)
    print("✅ STEP 1 AUDIT COMPLETE — VERDICT")
    print("="*70)
    
    proceed = True
    warnings = []
    
    if balance_ratio < 0.67:
        warnings.append("⚠️  Class imbalance may cause instability")
    
    if fold_issues:
        warnings.append(f"⚠️  {len(fold_issues)} folds have inadequate samples")
        proceed = False
    
    if returns is not None and noise_pct > 50:
        warnings.append("⚠️  Excessive noise in target - consider filtering")
    
    if warnings:
        print("\n".join(warnings))
        if not proceed:
            print("\n🚨 DO NOT PROCEED TO STEP 2 - Fix fold issues first")
    else:
        print("✅ All checks passed - target is structurally sound")
        print("✅ Ready to proceed to STEP 2 (Feature Semantics)")
    
    # Save audit report
    with open('data/step1_audit_report.json', 'w') as f:
        json.dump({
            'target_col': target_col,
            'audit_results': audit_results,
            'warnings': warnings,
            'proceed': proceed
        }, f, indent=2)
    
    print("\n📄 Audit report saved to: data/step1_audit_report.json")
    
    return audit_results, proceed

# ========== DATA LOADING PIPELINE ==========
def fetch_dune(qid, cache):
    headers = {"x-dune-api-key": DUNE_API_KEY}
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - timedelta(1)
    
    if os.path.exists(cache):
        with open(cache) as f:
            c = json.load(f)
        df_cached = pd.DataFrame(c["data"])
        df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
        last_date = pd.to_datetime(c["last_block_date"], utc=True)
        
        if last_date >= yesterday:
            print(f"✅ {os.path.basename(cache)} current ({last_date.date()})")
            return df_cached
        
        print(f"🔄 {os.path.basename(cache)}: fetching {(today - last_date).days} new days")
    else:
        df_cached = pd.DataFrame()
        print(f"🆕 {os.path.basename(cache)}: full fetch")
    
    resp = requests.post(
        f"https://api.dune.com/api/v1/query/{qid}/execute",
        headers=headers,
        timeout=30
    ).json()
    
    if "execution_id" not in resp:
        raise RuntimeError(f"Dune API error: {resp}")
    
    eid = resp["execution_id"]
    
    for _ in range(60):
        status = requests.get(
            f"https://api.dune.com/api/v1/execution/{eid}/status",
            headers=headers
        ).json()["state"]
        
        if status == "QUERY_STATE_COMPLETED":
            break
        if status == "QUERY_STATE_FAILED":
            raise RuntimeError("Query failed")
        time.sleep(10)
    
    result = requests.get(
        f"https://api.dune.com/api/v1/execution/{eid}/results",
        headers=headers
    ).json()["result"]["rows"]
    
    df_new = pd.DataFrame(result)
    if df_new.empty:
        return df_cached
    
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    df = pd.concat([
        df_cached,
        df_new[df_new["block_date"] < today]
    ]).drop_duplicates("block_date", keep="last").sort_values("block_date").reset_index(drop=True)
    
    with open(cache, "w") as f:
        json.dump({
            "last_block_date": df["block_date"].max().strftime("%Y-%m-%d"),
            "data": json.loads(df.to_json(orient="records", date_format="iso"))
        }, f)
    
    new_rows = len(df_new[df_new["block_date"] < today])
    print(f"✅ {os.path.basename(cache)}: {len(df)} rows (+{new_rows} new)")
    return df

def to_utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def fetch_cg_chunked(cg_id, start, end, key=None, days=30):
    url = "https://pro-api.coingecko.com/api/v3" if key else "https://api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key} if key else {}
    
    start_dt, end_dt = to_utc(start), to_utc(end) + pd.Timedelta(days=1)
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {
            "vs_currency": "usd",
            "from": int(curr.timestamp()),
            "to": int(next_dt.timestamp())
        }
        
        for attempt in range(3):
            try:
                r = requests.get(
                    f"{url}/coins/{cg_id}/market_chart/range",
                    params=params, headers=headers, timeout=30
                )
                r.raise_for_status()
                prices = r.json().get("prices", [])
                all_prices.extend(prices)
                print(f"📥 {cg_id}: {curr.date()} → {next_dt.date()} ({len(prices)} pts)")
                time.sleep(0.3)
                break
            except Exception as e:
                if attempt == 2: raise
                print(f"⚠️ Retry {attempt + 1}/3 ({e})")
                time.sleep(5)
        
        curr = next_dt
    
    if not all_prices:
        return pd.DataFrame(columns=["date", "price"])
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    df = df.groupby("date", as_index=False)["price"].mean().sort_values("date")
    
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D", tz="UTC")
    df = df.set_index("date").reindex(full_range).rename_axis("date").reset_index()
    
    return df

def get_price(sym, cg_id, start, end, key=None):
    cache = f"data/price_cache/{sym}.csv"
    
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    start, end = to_utc(start), min(to_utc(end), yesterday)
    
    if start > end:
        return pd.DataFrame(columns=["date", f"{sym}_price"])
    
    if os.path.exists(cache):
        df = pd.read_csv(cache, parse_dates=["date"])
        df["date"] = df["date"].apply(to_utc)
        last_cached = df["date"].max()
        
        if last_cached >= end:
            print(f"✅ {sym.upper()} cache current ({last_cached.date()})")
            return df
        
        fetch_start = last_cached + pd.Timedelta(days=1)
        print(f"🔄 {sym.upper()}: fetching {fetch_start.date()} → {end.date()}")
        
        new = fetch_cg_chunked(cg_id, fetch_start, end, key)
        if not new.empty:
            new = new.rename(columns={"price": f"{sym}_price"})
            df = pd.concat([df, new]).drop_duplicates("date", keep="last").sort_values("date").reset_index(drop=True)
    else:
        print(f"📦 {sym.upper()}: full fetch {start.date()} → {end.date()}")
        df = fetch_cg_chunked(cg_id, start, end, key)
        if not df.empty:
            df = df.rename(columns={"price": f"{sym}_price"})
    
    df.to_csv(cache, index=False)
    print(f"✅ {sym.upper()} saved (through {df['date'].max().date()})")
    return df

# ========== FEATURE ENGINEERING ==========
def rolling_zscore(s, w=90):
    return (s - s.rolling(w).mean()) / s.rolling(w).std()

def add_price_features(df, price_col, prefix):
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    for lag in [1, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std()
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std()
    
    ret = df[f'{prefix}_log_return']
    gains = ret.where(ret > 0, 0).rolling(14).mean()
    losses = -ret.where(ret < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = 100 - (100 / (1 + gains / (losses + 1e-10)))
    
    return df

def engineer_features(df):
    df = df.sort_values('block_date').reset_index(drop=True)
    
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean()
    
    df['eth_btc_corr_30d'] = (
        df['eth_log_return'].shift(1)
        .rolling(30, min_periods=20)
        .corr(df['btc_log_return'].shift(1))
    )
    
    onchain_raw = {
        'whale_tx_count': 'whale_tx_zscore_90d',
        'tx_per_active': 'tx_per_active_zscore_90d',
        'eth_burned': 'eth_burned_zscore_90d'
    }
    
    for raw_col, zscore_col in onchain_raw.items():
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore(df[raw_col], 90)
    
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3)
    
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    return df

# ========== TARGET CREATION ==========
def create_targets(df, horizons=[1, 2, 3]):
    df = df.sort_values('block_date').reset_index(drop=True)
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    
    for h in horizons:
        df[f'return_t{h}'] = df['eth_log_return'].rolling(h).sum().shift(-h)
        df[f'direction_t{h}'] = (df[f'return_t{h}'] > 0).astype(int)
    
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    return df

# ========== ABLATION STUDY ==========
def get_available_features(X, feature_list):
    return [f for f in feature_list if f in X.columns]

def ablation_study(X, y, n_splits=5):
    feature_groups = {
        'price': get_available_features(X, PRICE),
        'onchain': get_available_features(X, ONCHAIN),
        'hybrid': get_available_features(X, HYBRID)
    }
    
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=30)
    results = []
    
    for name, features in feature_groups.items():
        if not features:
            print(f"⚠️ No features for {name}")
            continue
        
        print(f"\n{'='*60}")
        print(f"{name.upper()}: {len(features)} features")
        print(f"{'='*60}")
        
        X_sub = X[features].fillna(method='ffill').fillna(0)
        
        base = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
        model = CalibratedClassifierCV(base, cv=3, method='sigmoid')
        
        fold_metrics = []
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X_sub), 1):
            X_tr, X_te = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
            y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
            
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            y_prob = model.predict_proba(X_te)[:, 1]
            
            m = {
                'acc': accuracy_score(y_te, y_pred),
                'auc': roc_auc_score(y_te, y_prob),
                'prec_up': precision_score(y_te, y_pred, pos_label=1, zero_division=0),
                'prec_down': precision_score(y_te, y_pred, pos_label=0, zero_division=0)
            }
            fold_metrics.append(m)
            print(f"  Fold {fold}: Acc={m['acc']:.3f}, AUC={m['auc']:.3f}, P↑={m['prec_up']:.3f}, P↓={m['prec_down']:.3f}")
        
        avg = {k: np.mean([m[k] for m in fold_metrics]) for k in fold_metrics[0].keys()}
        std = {k: np.std([m[k] for m in fold_metrics]) for k in fold_metrics[0].keys()}
        
        print(f"  → Avg: Acc={avg['acc']:.3f}±{std['acc']:.3f}, AUC={avg['auc']:.3f}±{std['auc']:.3f}")
        results.append({'feature_set': name, **avg})
    
    return pd.DataFrame(results), feature_groups

# ========== VETO SYSTEM ==========
class VetoDecisionEngine:
    def __init__(self, price_model, onchain_model, min_conf=0.10, price_thresh=0.55):
        self.price_model = price_model
        self.onchain_model = onchain_model
        self.min_conf = min_conf
        self.price_thresh = price_thresh
    
    def predict(self, price_feat, onchain_feat):
        price_prob_up = self.price_model.predict_proba([price_feat])[0, 1]
        onchain_prob_up = self.onchain_model.predict_proba([onchain_feat])[0, 1]
        
        price_prob_down = 1 - price_prob_up
        onchain_prob_down = 1 - onchain_prob_up
        price_conf = abs(price_prob_up - 0.5)
        
        if (price_prob_up > self.price_thresh and 
            onchain_prob_down < 0.50 and
            price_conf > self.min_conf):
            return 'LONG'
        
        if (price_prob_down > self.price_thresh and 
            onchain_prob_up < 0.50 and
            price_conf > self.min_conf):
            return 'SHORT'
        
        return 'NO_TRADE'

# ========== MAIN DAILY PIPELINE ==========
def run_daily_pipeline():
    print("="*70)
    print("ETH WHALE ML PIPELINE - WITH STEP 1 TARGET AUDIT")
    print("="*70)
    
    # 1️⃣ LOAD DATA
    print("\n📂 LOADING DATA...")
    datasets = {}
    for name, (qid, cache, output) in QUERIES.items():
        datasets[name] = fetch_dune(qid, cache)
        datasets[name].to_csv(output, index=False)
        time.sleep(0.5)
    
    df_whales = datasets["whales"]
    df_market_intent = datasets["market_intent"]
    
    min_date = min(df_whales["block_date"].min(), df_market_intent["block_date"].min()) - pd.Timedelta(days=100)
    max_date = max(df_whales["block_date"].max(), df_market_intent["block_date"].max())
    
    df_btc = get_price("btc", "bitcoin", min_date, max_date, COINGECKO_API_KEY)
    df_eth = get_price("eth", "ethereum", min_date, max_date, COINGECKO_API_KEY)
    
    # 2️⃣ MERGE
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    df = pd.merge(df_whales, df_prices, left_on='block_date', right_on='date', how='left').drop(columns=['date'])
    df = pd.merge(df, df_market_intent, on='block_date', how='left', suffixes=('', '_intent'))
    df.to_csv('data/merged_ml_dataset.csv', index=False)
    
    # 3️⃣ ENGINEER FEATURES
    print("\n" + "="*70)
    print("FEATURE ENGINEERING")
    print("="*70)
    df = engineer_features(df)
    df = create_targets(df, horizons=[1, 2, 3])
    df.to_csv('data/features_engineered.csv', index=False)
    
    # 🔍 STEP 1: TARGET AUDIT (NEW)
    audit_results, proceed = audit_target_structure(df, target_col='direction_t2', n_splits=5)
    
    if not proceed:
        print("\n🛑 Pipeline halted due to target audit failures")
        print("   Fix identified issues before proceeding")
        return None, None
    
    # 4️⃣ ABLATION (T+2)
    print("\n" + "="*70)
    print("ABLATION STUDY (T+2)")
    print("="*70)
    
    target_cols = ['direction_t2', 'direction_t3', 'return_t2', 'return_t3', 'block_date']
    X = df.drop(columns=target_cols, errors='ignore').dropna()
    y = df.loc[X.index, 'direction_t2']
    
    results, feature_groups = ablation_study(X, y)
    print("\n" + results.to_string(index=False, float_format='%.4f'))
    results.to_csv('data/ablation_t2.csv', index=False)
    
    # 5️⃣ TRAIN MODELS
    print("\n" + "="*70)
    print("TRAINING VETO SYSTEM")
    print("="*70)
    
    X_price = X[feature_groups['price']].fillna(method='ffill').fillna(0)
    X_onchain = X[feature_groups['onchain']].fillna(method='ffill').fillna(0)
    
    price_model = CalibratedClassifierCV(
        GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
        cv=3, method='sigmoid'
    ).fit(X_price, y)
    
    onchain_model = CalibratedClassifierCV(
        LogisticRegression(max_iter=1000),
        cv=3, method='sigmoid'
    ).fit(X_onchain, y)
    
    joblib.dump(price_model, 'data/price_model.pkl')
    joblib.dump(onchain_model, 'data/onchain_model.pkl')
    print("✅ Models saved")
    
    # 6️⃣ TEST DECISION
    engine = VetoDecisionEngine(price_model, onchain_model)
    decision = engine.predict(X_price.iloc[-1].values, X_onchain.iloc[-1].values)
    
    print(f"\n{'='*70}")
    print(f"🎯 LATEST DECISION: {decision}")
    print(f"{'='*70}")
    
    return engine, results

if __name__ == "__main__":
    engine, results = run_daily_pipeline()
    if engine:
        print("\n✅ Daily pipeline complete!")
    else:
        print("\n⚠️  Pipeline stopped at Step 1 audit")

ETH WHALE ML PIPELINE - WITH STEP 1 TARGET AUDIT

📂 LOADING DATA...
✅ dune_whales_cache.json current (2025-12-28)
✅ dune_intent_cache.json current (2025-12-28)
✅ BTC cache current (2025-12-28)
✅ ETH cache current (2025-12-28)

FEATURE ENGINEERING

STEP 1 — FORMAL DIRECTIONAL TARGET AUDIT

1️⃣ GLOBAL CLASS BALANCE
----------------------------------------------------------------------
DOWN (0): 48.6%
UP   (1): 51.4%

Balance Status: ✅ EXCELLENT

2️⃣ PER-FOLD CLASS BALANCE (CRITICAL)
----------------------------------------------------------------------
Fold 1: DOWN=14, UP=16 ✅
Fold 2: DOWN=14, UP=16 ✅
Fold 3: DOWN=12, UP=18 ✅
Fold 4: DOWN=19, UP=11 ✅
Fold 5: DOWN=18, UP=12 ✅

✅ All folds have adequate samples

3️⃣ BASE-RATE BENCHMARK (Reality Check)
----------------------------------------------------------------------
Baseline Accuracy (majority class): 0.5139

⚠️  Any model scoring < 0.5139 is statistically useless

4️⃣ MOVE MAGNITUDE AUDIT (Noise Detection)
---------------------------

In [ ]:
"""
ETH Whale Activity ML Pipeline - With Step 1 Target Audit
Complete Daily Pipeline with Formal Directional Target Verification
"""

import os
import time
import json
import requests
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score
from dotenv import load_dotenv
import joblib
import warnings
warnings.filterwarnings('ignore')

# ========== CONFIGURATION ==========
load_dotenv()
DUNE_API_KEY = os.getenv("DUNE_WHALES_API")
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")
os.makedirs("data", exist_ok=True)
os.makedirs("data/price_cache", exist_ok=True)

QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

PRICE = [
    'eth_ret_lag1','eth_ret_lag3','eth_ret_lag7',
    'eth_vol7','eth_vol30','eth_rsi',
    'btc_ret_lag1','btc_ret_lag3','btc_ret_lag7',
    'btc_vol7','btc_vol30','btc_rsi',
    'eth_btc_ratio','eth_btc_ratio_ma7','eth_btc_corr_30d'
]

ONCHAIN = [
    'whale_tx_zscore_90d','whale_volume_ratio',
    'whale_volume_ratio_delta_1d','whale_volume_ratio_delta_3d',
    'exchange_flow_share','net_exchange_flow_ratio',
    'whale_exchange_flow_ratio','whale_exchange_asymmetry',
    'tx_per_active_zscore_90d','eth_burned_zscore_90d'
]

HYBRID = PRICE + ONCHAIN

# ========== STEP 1: TARGET AUDIT FUNCTIONS ==========

def audit_target_structure(df, target_col='direction_t2', n_splits=5):
    """
    STEP 1 — FORMAL DIRECTIONAL TARGET AUDIT
    Verifies target is learnable, balanced, and correctly evaluated
    """
    print("\n" + "="*70)
    print("STEP 1 — FORMAL DIRECTIONAL TARGET AUDIT")
    print("="*70)
    
    # Prepare data
    target_cols = ['direction_t2', 'direction_t3', 'return_t2', 'return_t3', 'block_date']
    X = df.drop(columns=target_cols, errors='ignore').dropna()
    y = df.loc[X.index, target_col]
    
    # Also get returns for magnitude analysis
    returns = df.loc[X.index, 'return_t2'] if 'return_t2' in df.columns else None
    
    audit_results = {}
    
    # 1️⃣ GLOBAL CLASS BALANCE
    print("\n1️⃣ GLOBAL CLASS BALANCE")
    print("-" * 70)
    
    class_dist = y.value_counts(normalize=True).sort_index()
    up_pct = class_dist.get(1, 0) * 100
    down_pct = class_dist.get(0, 0) * 100
    
    print(f"DOWN (0): {down_pct:.1f}%")
    print(f"UP   (1): {up_pct:.1f}%")
    
    # Assess balance
    balance_ratio = min(up_pct, down_pct) / max(up_pct, down_pct)
    if balance_ratio >= 0.80:
        status = "✅ EXCELLENT"
    elif balance_ratio >= 0.67:
        status = "⚠️  MANAGEABLE"
    elif balance_ratio >= 0.54:
        status = "⚠️  BIAS RISK"
    else:
        status = "🚨 CRITICAL - Model will collapse to majority"
    
    print(f"\nBalance Status: {status}")
    audit_results['global_balance'] = {'up_pct': up_pct, 'down_pct': down_pct, 'status': status}
    
    # 2️⃣ PER-FOLD CLASS BALANCE
    print("\n2️⃣ PER-FOLD CLASS BALANCE (CRITICAL)")
    print("-" * 70)
    
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=30)
    fold_issues = []
    
    for fold, (_, test_idx) in enumerate(tscv.split(X), 1):
        y_fold = y.iloc[test_idx]
        counts = y_fold.value_counts().sort_index()
        
        down_count = counts.get(0, 0)
        up_count = counts.get(1, 0)
        
        # Check for critical issues
        issue = None
        if down_count < 10:
            issue = f"🚨 <10 DOWN samples ({down_count})"
        elif up_count < 10:
            issue = f"🚨 <10 UP samples ({up_count})"
        elif down_count == 0 or up_count == 0:
            issue = "🚨 SINGLE CLASS FOLD"
        
        status_icon = "🚨" if issue else "✅"
        print(f"Fold {fold}: DOWN={down_count:2d}, UP={up_count:2d} {status_icon}")
        
        if issue:
            fold_issues.append((fold, issue))
            print(f"         {issue}")
    
    if fold_issues:
        print(f"\n⚠️  {len(fold_issues)} folds have critical issues")
        print("   This explains P↓=0.000 or P↓=1.000 instability")
    else:
        print("\n✅ All folds have adequate samples")
    
    audit_results['fold_balance'] = {'issues': fold_issues, 'n_folds': n_splits}
    
    # 3️⃣ BASE-RATE BENCHMARK
    print("\n3️⃣ BASE-RATE BENCHMARK (Reality Check)")
    print("-" * 70)
    
    baseline_acc = max(y.mean(), 1 - y.mean())
    print(f"Baseline Accuracy (majority class): {baseline_acc:.4f}")
    print(f"\n⚠️  Any model scoring < {baseline_acc:.4f} is statistically useless")
    
    audit_results['baseline'] = baseline_acc
    
    # 4️⃣ MOVE MAGNITUDE AUDIT
    if returns is not None:
        print("\n4️⃣ MOVE MAGNITUDE AUDIT (Noise Detection)")
        print("-" * 70)
        
        abs_returns = returns.abs()
        print("\nReturn Magnitude Distribution:")
        print(abs_returns.describe().to_string())
        
        # Check for coin-flip territory
        noise_threshold = 0.002
        noise_pct = (abs_returns < noise_threshold).mean() * 100
        
        print(f"\nMoves < {noise_threshold:.1%}: {noise_pct:.1f}%")
        
        if noise_pct > 50:
            print("🚨 CRITICAL: >50% moves are noise - predicting coin flips")
        elif noise_pct > 40:
            print("⚠️  WARNING: High noise ratio - consider volatility gating")
        else:
            print("✅ Acceptable signal-to-noise ratio")
        
        audit_results['noise'] = {'pct_small_moves': noise_pct, 'threshold': noise_threshold}
    
    # 5️⃣ LABEL STABILITY TEST
    if all(f'direction_t{h}' in df.columns for h in [1, 2, 3]):
        print("\n5️⃣ LABEL STABILITY TEST (T+1 vs T+2 vs T+3)")
        print("-" * 70)
        
        valid_idx = df[['direction_t1', 'direction_t2', 'direction_t3']].dropna().index
        
        align_12 = (df.loc[valid_idx, 'direction_t1'] == df.loc[valid_idx, 'direction_t2']).mean()
        align_13 = (df.loc[valid_idx, 'direction_t1'] == df.loc[valid_idx, 'direction_t3']).mean()
        align_23 = (df.loc[valid_idx, 'direction_t2'] == df.loc[valid_idx, 'direction_t3']).mean()
        
        print(f"T+1 vs T+2 agreement: {align_12:.1%}")
        print(f"T+1 vs T+3 agreement: {align_13:.1%}")
        print(f"T+2 vs T+3 agreement: {align_23:.1%}")
        
        if min(align_12, align_13, align_23) < 0.60:
            print("\n⚠️  WARNING: Low directional stability (<60%)")
            print("   This explains why T+2 may outperform T+1")
        else:
            print("\n✅ Acceptable directional stability")
        
        audit_results['stability'] = {
            't1_t2': align_12,
            't1_t3': align_13,
            't2_t3': align_23
        }
    
    # ✅ FINAL VERDICT
    print("\n" + "="*70)
    print("✅ STEP 1 AUDIT COMPLETE — VERDICT")
    print("="*70)
    
    proceed = True
    warnings = []
    
    if balance_ratio < 0.67:
        warnings.append("⚠️  Class imbalance may cause instability")
    
    if fold_issues:
        warnings.append(f"⚠️  {len(fold_issues)} folds have inadequate samples")
        proceed = False
    
    if returns is not None and noise_pct > 50:
        warnings.append("⚠️  Excessive noise in target - consider filtering")
    
    if warnings:
        print("\n".join(warnings))
        if not proceed:
            print("\n🚨 DO NOT PROCEED TO STEP 2 - Fix fold issues first")
    else:
        print("✅ All checks passed - target is structurally sound")
        print("✅ Ready to proceed to STEP 2 (Feature Semantics)")
    
    # Save audit report
    with open('data/step1_audit_report.json', 'w') as f:
        json.dump({
            'target_col': target_col,
            'audit_results': audit_results,
            'warnings': warnings,
            'proceed': proceed
        }, f, indent=2)
    
    print("\n📄 Audit report saved to: data/step1_audit_report.json")
    
    return audit_results, proceed

# ========== STEP 2: FEATURE SEMANTICS AUDIT ==========

def compute_feature_direction_correlation(df, features, target='direction_t2'):
    """
    Compute directional correlation for each feature
    Positive = bullish, Negative = bearish, Near-zero = noise
    """
    valid_idx = df[features + [target]].dropna().index
    X = df.loc[valid_idx, features]
    y = df.loc[valid_idx, target]
    
    correlations = []
    for feat in features:
        # Spearman correlation (robust to outliers)
        corr = X[feat].corr(y, method='spearman')
        
        # Also check if feature distinguishes UP vs DOWN
        up_mean = X.loc[y == 1, feat].mean()
        down_mean = X.loc[y == 0, feat].mean()
        
        # Effect size (Cohen's d)
        pooled_std = np.sqrt((X.loc[y == 1, feat].var() + X.loc[y == 0, feat].var()) / 2)
        cohens_d = (up_mean - down_mean) / (pooled_std + 1e-10)
        
        correlations.append({
            'feature': feat,
            'spearman_corr': corr,
            'cohens_d': cohens_d,
            'up_mean': up_mean,
            'down_mean': down_mean
        })
    
    return pd.DataFrame(correlations)

def classify_feature_semantics(corr_df, corr_thresh=0.05, effect_thresh=0.10):
    """
    Classify features as BULLISH, BEARISH, or NOISE
    """
    corr_df['signal_strength'] = corr_df['spearman_corr'].abs()
    corr_df['effect_strength'] = corr_df['cohens_d'].abs()
    
    # Classification logic
    def classify(row):
        if row['signal_strength'] < corr_thresh and row['effect_strength'] < effect_thresh:
            return 'NOISE'
        elif row['spearman_corr'] > corr_thresh or row['cohens_d'] > effect_thresh:
            return 'BULLISH'
        elif row['spearman_corr'] < -corr_thresh or row['cohens_d'] < -effect_thresh:
            return 'BEARISH'
        else:
            return 'WEAK'
    
    corr_df['semantic_class'] = corr_df.apply(classify, axis=1)
    return corr_df

def audit_feature_semantics(df, feature_groups, target='direction_t2'):
    """
    STEP 2 — DIRECTIONAL FEATURE SEMANTICS AUDIT
    Identifies which features are bullish, bearish, or noise
    """
    print("\n" + "="*70)
    print("STEP 2 — DIRECTIONAL FEATURE SEMANTICS AUDIT")
    print("="*70)
    
    all_results = {}
    
    for group_name, features in feature_groups.items():
        if not features:
            continue
            
        print(f"\n{'='*70}")
        print(f"{group_name.upper()}: Analyzing {len(features)} features")
        print(f"{'='*70}")
        
        # Compute correlations
        corr_df = compute_feature_direction_correlation(df, features, target)
        corr_df = classify_feature_semantics(corr_df)
        
        # Sort by signal strength
        corr_df = corr_df.sort_values('signal_strength', ascending=False)
        
        # Summary by semantic class
        semantic_summary = corr_df['semantic_class'].value_counts()
        
        print(f"\n📊 SEMANTIC CLASSIFICATION:")
        for sem_class in ['BULLISH', 'BEARISH', 'WEAK', 'NOISE']:
            count = semantic_summary.get(sem_class, 0)
            pct = (count / len(features)) * 100
            icon = {'BULLISH': '🟢', 'BEARISH': '🔴', 'WEAK': '🟡', 'NOISE': '⚪'}[sem_class]
            print(f"  {icon} {sem_class:8s}: {count:2d} features ({pct:5.1f}%)")
        
        # Show top signals
        print(f"\n🔝 TOP 10 SIGNALS (by strength):")
        print("-" * 70)
        
        top_10 = corr_df.head(10)
        for _, row in top_10.iterrows():
            icon = {'BULLISH': '🟢', 'BEARISH': '🔴', 'WEAK': '🟡', 'NOISE': '⚪'}[row['semantic_class']]
            print(f"{icon} {row['feature']:30s} | ρ={row['spearman_corr']:+.3f} | d={row['cohens_d']:+.3f} | {row['semantic_class']}")
        
        # Flag problematic patterns
        noise_pct = (semantic_summary.get('NOISE', 0) / len(features)) * 100
        weak_pct = (semantic_summary.get('WEAK', 0) / len(features)) * 100
        
        if noise_pct > 50:
            print(f"\n🚨 WARNING: {noise_pct:.0f}% features are NOISE - remove them!")
        elif noise_pct + weak_pct > 60:
            print(f"\n⚠️  WARNING: {noise_pct + weak_pct:.0f}% features are weak/noise")
        
        all_results[group_name] = corr_df
    
    # Cross-group semantic analysis
    print(f"\n{'='*70}")
    print("CROSS-GROUP SEMANTIC COHERENCE")
    print(f"{'='*70}")
    
    # Check if on-chain contradicts price
    if 'price' in all_results and 'onchain' in all_results:
        price_bullish = set(all_results['price'][all_results['price']['semantic_class'] == 'BULLISH']['feature'])
        onchain_bearish = set(all_results['onchain'][all_results['onchain']['semantic_class'] == 'BEARISH']['feature'])
        
        price_avg_corr = all_results['price']['spearman_corr'].mean()
        onchain_avg_corr = all_results['onchain']['spearman_corr'].mean()
        
        print(f"\nPrice features avg correlation:    {price_avg_corr:+.3f}")
        print(f"On-chain features avg correlation: {onchain_avg_corr:+.3f}")
        
        if price_avg_corr > 0 and onchain_avg_corr < 0:
            print("\n🚨 CONTRADICTION DETECTED:")
            print("   Price features are bullish while on-chain is bearish")
            print("   → This explains why HYBRID < PRICE in ablation")
        elif abs(onchain_avg_corr) < 0.02:
            print("\n⚠️  On-chain features have weak directional signal")
            print("   → They add noise, not information")
    
    # Generate cleaned feature sets
    print(f"\n{'='*70}")
    print("✅ RECOMMENDED FEATURE SETS (cleaned)")
    print(f"{'='*70}")
    
    cleaned_features = {}
    for group_name, corr_df in all_results.items():
        # Keep only BULLISH and BEARISH (remove NOISE and WEAK)
        strong_features = corr_df[corr_df['semantic_class'].isin(['BULLISH', 'BEARISH'])]['feature'].tolist()
        cleaned_features[f'{group_name}_cleaned'] = strong_features
        
        removed = len(corr_df) - len(strong_features)
        print(f"\n{group_name.upper()}_CLEANED: {len(strong_features)} features (removed {removed} weak/noise)")
    
    # Save detailed report with fallback mechanism
    try:
        # Try to save as Excel
        with pd.ExcelWriter('data/step2_feature_semantics.xlsx', engine='openpyxl') as writer:
            for group_name, corr_df in all_results.items():
                corr_df.to_excel(writer, sheet_name=group_name, index=False)
        print(f"\n📄 Detailed report saved to: data/step2_feature_semantics.xlsx")
    except ImportError:
        # Fall back to CSV if openpyxl is not available
        print("\n⚠️  openpyxl not found, saving as CSV files instead...")
        for group_name, corr_df in all_results.items():
            corr_df.to_csv(f'data/step2_feature_semantics_{group_name}.csv', index=False)
        print(f"📄 Detailed reports saved to: data/step2_feature_semantics_*.csv")
    except Exception as e:
        print(f"\n⚠️  Could not save report: {e}")
    
    return all_results, cleaned_features

# ========== STEP 2: ENHANCED FEATURE ENGINEERING ==========

def create_semantic_features(df):
    """
    Create explicit accumulation/distribution/panic features
    """
    print("\n🔧 Creating semantic whale behavior features...")
    
    # ACCUMULATION SIGNALS (bullish)
    if 'whale_volume_ratio' in df.columns:
        # Whales buying MORE than usual
        df['whale_accumulation'] = (
            (df['whale_volume_ratio'] > df['whale_volume_ratio'].rolling(30).mean()) &
            (df['whale_volume_ratio_delta_1d'] > 0)
        ).astype(int)
    
    if 'net_exchange_flow_ratio' in df.columns:
        # Net outflow from exchanges (accumulation)
        df['exchange_outflow_strength'] = -df['net_exchange_flow_ratio'].clip(lower=-1, upper=0)
    
    # DISTRIBUTION SIGNALS (bearish)
    if 'whale_exchange_flow_ratio' in df.columns:
        # Whales moving TO exchanges (distribution)
        df['whale_distribution'] = (
            df['whale_exchange_flow_ratio'] > df['whale_exchange_flow_ratio'].rolling(30).mean()
        ).astype(int)
    
    if 'exchange_flow_share' in df.columns:
        # High exchange inflow share (selling pressure)
        df['exchange_inflow_pressure'] = df['exchange_flow_share'].clip(upper=1)
    
    # PANIC/CONVICTION SIGNALS
    if 'whale_tx_zscore_90d' in df.columns and 'eth_vol7' in df.columns:
        # High activity + high volatility = panic
        df['whale_panic_index'] = (
            (df['whale_tx_zscore_90d'] > 1.5) & 
            (df['eth_vol7'] > df['eth_vol7'].rolling(30).mean())
        ).astype(int)
        
        # High activity + low volatility = conviction
        df['whale_conviction_index'] = (
            (df['whale_tx_zscore_90d'] > 1.5) & 
            (df['eth_vol7'] < df['eth_vol7'].rolling(30).mean())
        ).astype(int)
    
    # MOMENTUM ALIGNMENT
    if all(f in df.columns for f in ['eth_ret_lag1', 'whale_volume_ratio_delta_1d']):
        # Whales following momentum (chasing)
        df['whale_momentum_chase'] = (
            (df['eth_ret_lag1'] > 0) & (df['whale_volume_ratio_delta_1d'] > 0)
        ).astype(int)
        
        # Whales contrarian (buying dips)
        df['whale_contrarian'] = (
            (df['eth_ret_lag1'] < 0) & (df['whale_volume_ratio_delta_1d'] > 0)
        ).astype(int)
    
    new_features = [
        'whale_accumulation', 'exchange_outflow_strength',
        'whale_distribution', 'exchange_inflow_pressure',
        'whale_panic_index', 'whale_conviction_index',
        'whale_momentum_chase', 'whale_contrarian'
    ]
    
    created = [f for f in new_features if f in df.columns]
    print(f"✅ Created {len(created)} semantic features: {created}")
    
    return df

# ========== MAIN PIPELINE WITH STEP 2 ==========
def fetch_dune(qid, cache):
    headers = {"x-dune-api-key": DUNE_API_KEY}
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - timedelta(1)
    
    if os.path.exists(cache):
        with open(cache) as f:
            c = json.load(f)
        df_cached = pd.DataFrame(c["data"])
        df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
        last_date = pd.to_datetime(c["last_block_date"], utc=True)
        
        if last_date >= yesterday:
            print(f"✅ {os.path.basename(cache)} current ({last_date.date()})")
            return df_cached
        
        print(f"🔄 {os.path.basename(cache)}: fetching {(today - last_date).days} new days")
    else:
        df_cached = pd.DataFrame()
        print(f"🆕 {os.path.basename(cache)}: full fetch")
    
    resp = requests.post(
        f"https://api.dune.com/api/v1/query/{qid}/execute",
        headers=headers,
        timeout=30
    ).json()
    
    if "execution_id" not in resp:
        raise RuntimeError(f"Dune API error: {resp}")
    
    eid = resp["execution_id"]
    
    for _ in range(60):
        status = requests.get(
            f"https://api.dune.com/api/v1/execution/{eid}/status",
            headers=headers
        ).json()["state"]
        
        if status == "QUERY_STATE_COMPLETED":
            break
        if status == "QUERY_STATE_FAILED":
            raise RuntimeError("Query failed")
        time.sleep(10)
    
    result = requests.get(
        f"https://api.dune.com/api/v1/execution/{eid}/results",
        headers=headers
    ).json()["result"]["rows"]
    
    df_new = pd.DataFrame(result)
    if df_new.empty:
        return df_cached
    
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    df = pd.concat([
        df_cached,
        df_new[df_new["block_date"] < today]
    ]).drop_duplicates("block_date", keep="last").sort_values("block_date").reset_index(drop=True)
    
    with open(cache, "w") as f:
        json.dump({
            "last_block_date": df["block_date"].max().strftime("%Y-%m-%d"),
            "data": json.loads(df.to_json(orient="records", date_format="iso"))
        }, f)
    
    new_rows = len(df_new[df_new["block_date"] < today])
    print(f"✅ {os.path.basename(cache)}: {len(df)} rows (+{new_rows} new)")
    return df

def to_utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def fetch_cg_chunked(cg_id, start, end, key=None, days=30):
    url = "https://pro-api.coingecko.com/api/v3" if key else "https://api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key} if key else {}
    
    start_dt, end_dt = to_utc(start), to_utc(end) + pd.Timedelta(days=1)
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {
            "vs_currency": "usd",
            "from": int(curr.timestamp()),
            "to": int(next_dt.timestamp())
        }
        
        for attempt in range(3):
            try:
                r = requests.get(
                    f"{url}/coins/{cg_id}/market_chart/range",
                    params=params, headers=headers, timeout=30
                )
                r.raise_for_status()
                prices = r.json().get("prices", [])
                all_prices.extend(prices)
                print(f"📥 {cg_id}: {curr.date()} → {next_dt.date()} ({len(prices)} pts)")
                time.sleep(0.3)
                break
            except Exception as e:
                if attempt == 2: raise
                print(f"⚠️ Retry {attempt + 1}/3 ({e})")
                time.sleep(5)
        
        curr = next_dt
    
    if not all_prices:
        return pd.DataFrame(columns=["date", "price"])
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    df = df.groupby("date", as_index=False)["price"].mean().sort_values("date")
    
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D", tz="UTC")
    df = df.set_index("date").reindex(full_range).rename_axis("date").reset_index()
    
    return df

def get_price(sym, cg_id, start, end, key=None):
    cache = f"data/price_cache/{sym}.csv"
    
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    start, end = to_utc(start), min(to_utc(end), yesterday)
    
    if start > end:
        return pd.DataFrame(columns=["date", f"{sym}_price"])
    
    if os.path.exists(cache):
        df = pd.read_csv(cache, parse_dates=["date"])
        df["date"] = df["date"].apply(to_utc)
        last_cached = df["date"].max()
        
        if last_cached >= end:
            print(f"✅ {sym.upper()} cache current ({last_cached.date()})")
            return df
        
        fetch_start = last_cached + pd.Timedelta(days=1)
        print(f"🔄 {sym.upper()}: fetching {fetch_start.date()} → {end.date()}")
        
        new = fetch_cg_chunked(cg_id, fetch_start, end, key)
        if not new.empty:
            new = new.rename(columns={"price": f"{sym}_price"})
            df = pd.concat([df, new]).drop_duplicates("date", keep="last").sort_values("date").reset_index(drop=True)
    else:
        print(f"📦 {sym.upper()}: full fetch {start.date()} → {end.date()}")
        df = fetch_cg_chunked(cg_id, start, end, key)
        if not df.empty:
            df = df.rename(columns={"price": f"{sym}_price"})
    
    df.to_csv(cache, index=False)
    print(f"✅ {sym.upper()} saved (through {df['date'].max().date()})")
    return df

# ========== FEATURE ENGINEERING ==========
def rolling_zscore(s, w=90):
    return (s - s.rolling(w).mean()) / s.rolling(w).std()

def add_price_features(df, price_col, prefix):
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    for lag in [1, 3, 7]:
        df[f'{prefix}_ret_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7).std()
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30).std()
    
    ret = df[f'{prefix}_log_return']
    gains = ret.where(ret > 0, 0).rolling(14).mean()
    losses = -ret.where(ret < 0, 0).rolling(14).mean()
    df[f'{prefix}_rsi'] = 100 - (100 / (1 + gains / (losses + 1e-10)))
    
    return df

def engineer_features(df):
    df = df.sort_values('block_date').reset_index(drop=True)
    
    df = add_price_features(df, 'eth_price', 'eth')
    df = add_price_features(df, 'btc_price', 'btc')
    
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7).mean()
    
    df['eth_btc_corr_30d'] = (
        df['eth_log_return'].shift(1)
        .rolling(30, min_periods=20)
        .corr(df['btc_log_return'].shift(1))
    )
    
    onchain_raw = {
        'whale_tx_count': 'whale_tx_zscore_90d',
        'tx_per_active': 'tx_per_active_zscore_90d',
        'eth_burned': 'eth_burned_zscore_90d'
    }
    
    for raw_col, zscore_col in onchain_raw.items():
        if raw_col in df.columns:
            df[zscore_col] = rolling_zscore(df[raw_col], 90)
    
    if 'whale_volume_ratio' in df.columns:
        df['whale_volume_ratio_delta_1d'] = df['whale_volume_ratio'].diff(1)
        df['whale_volume_ratio_delta_3d'] = df['whale_volume_ratio'].diff(3)
    
    # Add semantic features
    df = create_semantic_features(df)
    
    df = df.drop(columns=['eth_log_return', 'btc_log_return'], errors='ignore')
    
    return df

# ========== TARGET CREATION ==========
def create_targets(df, horizons=[1, 2, 3]):
    df = df.sort_values('block_date').reset_index(drop=True)
    df['eth_log_return'] = np.log(df['eth_price'] / df['eth_price'].shift(1))
    
    for h in horizons:
        df[f'return_t{h}'] = df['eth_log_return'].rolling(h).sum().shift(-h)
        df[f'direction_t{h}'] = (df[f'return_t{h}'] > 0).astype(int)
    
    df = df.drop(columns=['eth_log_return'], errors='ignore')
    return df

# ========== ABLATION STUDY ==========
def get_available_features(X, feature_list):
    return [f for f in feature_list if f in X.columns]

def ablation_study(X, y, n_splits=5):
    feature_groups = {
        'price': get_available_features(X, PRICE),
        'onchain': get_available_features(X, ONCHAIN),
        'hybrid': get_available_features(X, HYBRID)
    }
    
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=30)
    results = []
    
    for name, features in feature_groups.items():
        if not features:
            print(f"⚠️ No features for {name}")
            continue
        
        print(f"\n{'='*60}")
        print(f"{name.upper()}: {len(features)} features")
        print(f"{'='*60}")
        
        X_sub = X[features].fillna(method='ffill').fillna(0)
        
        base = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
        model = CalibratedClassifierCV(base, cv=3, method='sigmoid')
        
        fold_metrics = []
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X_sub), 1):
            X_tr, X_te = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
            y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
            
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            y_prob = model.predict_proba(X_te)[:, 1]
            
            m = {
                'acc': accuracy_score(y_te, y_pred),
                'auc': roc_auc_score(y_te, y_prob),
                'prec_up': precision_score(y_te, y_pred, pos_label=1, zero_division=0),
                'prec_down': precision_score(y_te, y_pred, pos_label=0, zero_division=0)
            }
            fold_metrics.append(m)
            print(f"  Fold {fold}: Acc={m['acc']:.3f}, AUC={m['auc']:.3f}, P↑={m['prec_up']:.3f}, P↓={m['prec_down']:.3f}")
        
        avg = {k: np.mean([m[k] for m in fold_metrics]) for k in fold_metrics[0].keys()}
        std = {k: np.std([m[k] for m in fold_metrics]) for k in fold_metrics[0].keys()}
        
        print(f"  → Avg: Acc={avg['acc']:.3f}±{std['acc']:.3f}, AUC={avg['auc']:.3f}±{std['auc']:.3f}")
        results.append({'feature_set': name, **avg})
    
    return pd.DataFrame(results), feature_groups

# ========== VETO SYSTEM ==========
class VetoDecisionEngine:
    def __init__(self, price_model, onchain_model, min_conf=0.10, price_thresh=0.55):
        self.price_model = price_model
        self.onchain_model = onchain_model
        self.min_conf = min_conf
        self.price_thresh = price_thresh
    
    def predict(self, price_feat, onchain_feat):
        price_prob_up = self.price_model.predict_proba([price_feat])[0, 1]
        onchain_prob_up = self.onchain_model.predict_proba([onchain_feat])[0, 1]
        
        price_prob_down = 1 - price_prob_up
        onchain_prob_down = 1 - onchain_prob_up
        price_conf = abs(price_prob_up - 0.5)
        
        if (price_prob_up > self.price_thresh and 
            onchain_prob_down < 0.50 and
            price_conf > self.min_conf):
            return 'LONG'
        
        if (price_prob_down > self.price_thresh and 
            onchain_prob_up < 0.50 and
            price_conf > self.min_conf):
            return 'SHORT'
        
        return 'NO_TRADE'

# ========== MAIN DAILY PIPELINE ==========
def run_daily_pipeline():
    print("="*70)
    print("ETH WHALE ML PIPELINE - WITH STEP 1 & 2 AUDITS")
    print("="*70)
    
    # 1️⃣ LOAD DATA
    print("\n📂 LOADING DATA...")
    datasets = {}
    for name, (qid, cache, output) in QUERIES.items():
        datasets[name] = fetch_dune(qid, cache)
        datasets[name].to_csv(output, index=False)
        time.sleep(0.5)
    
    df_whales = datasets["whales"]
    df_market_intent = datasets["market_intent"]
    
    min_date = min(df_whales["block_date"].min(), df_market_intent["block_date"].min()) - pd.Timedelta(days=100)
    max_date = max(df_whales["block_date"].max(), df_market_intent["block_date"].max())
    
    df_btc = get_price("btc", "bitcoin", min_date, max_date, COINGECKO_API_KEY)
    df_eth = get_price("eth", "ethereum", min_date, max_date, COINGECKO_API_KEY)
    
    # 2️⃣ MERGE
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    df = pd.merge(df_whales, df_prices, left_on='block_date', right_on='date', how='left').drop(columns=['date'])
    df = pd.merge(df, df_market_intent, on='block_date', how='left', suffixes=('', '_intent'))
    df.to_csv('data/merged_ml_dataset.csv', index=False)
    
    # 3️⃣ ENGINEER FEATURES
    print("\n" + "="*70)
    print("FEATURE ENGINEERING")
    print("="*70)
    df = engineer_features(df)
    df = create_targets(df, horizons=[1, 2, 3])
    df.to_csv('data/features_engineered.csv', index=False)
    
    # 🔍 STEP 1: TARGET AUDIT (NEW)
    audit_results, proceed = audit_target_structure(df, target_col='direction_t2', n_splits=5)
    
    if not proceed:
        print("\n🛑 Pipeline halted due to target audit failures")
        print("   Fix identified issues before proceeding")
        return None, None
    
    # 🔍 STEP 2: FEATURE SEMANTICS AUDIT
    target_cols = ['direction_t1', 'direction_t2', 'direction_t3', 'return_t2', 'return_t3', 'block_date']
    X = df.drop(columns=target_cols, errors='ignore').dropna()
    
    # Get available features for each group
    initial_feature_groups = {
        'price': get_available_features(X, PRICE),
        'onchain': get_available_features(X, ONCHAIN),
    }
    
    # Add semantic features to onchain group
    semantic_feats = [
        'whale_accumulation', 'exchange_outflow_strength',
        'whale_distribution', 'exchange_inflow_pressure',
        'whale_panic_index', 'whale_conviction_index',
        'whale_momentum_chase', 'whale_contrarian'
    ]
    initial_feature_groups['onchain_semantic'] = get_available_features(X, semantic_feats)
    
    # Run semantic audit
    semantic_results, cleaned_features = audit_feature_semantics(
        df.loc[X.index], 
        initial_feature_groups,
        target='direction_t2'
    )
    
    # Use cleaned feature sets for modeling
    print(f"\n{'='*70}")
    print("REBUILDING FEATURE GROUPS WITH CLEANED FEATURES")
    print(f"{'='*70}")
    
    feature_groups = {}
    for group_name, features in cleaned_features.items():
        available = get_available_features(X, features)
        if available:
            feature_groups[group_name] = available
            print(f"✅ {group_name}: {len(available)} features")
    
    # Also keep original hybrid for comparison
    feature_groups['hybrid_original'] = get_available_features(X, HYBRID)
    
    # 4️⃣ ABLATION (T+2) WITH CLEANED FEATURES
    print("\n" + "="*70)
    print("ABLATION STUDY (T+2) - CLEANED FEATURES")
    print("="*70)
    
    y = df.loc[X.index, 'direction_t2']
    
    results, feature_groups = ablation_study(X, y)
    print("\n" + results.to_string(index=False, float_format='%.4f'))
    results.to_csv('data/ablation_t2.csv', index=False)
    
    # 5️⃣ TRAIN MODELS (use best cleaned set)
    print("\n" + "="*70)
    print("TRAINING VETO SYSTEM (CLEANED FEATURES)")
    print("="*70)
    
    # Use cleaned price and onchain features
    price_feats = feature_groups.get('price_cleaned', feature_groups.get('price', []))
    onchain_feats = feature_groups.get('onchain_cleaned', feature_groups.get('onchain', []))
    
    # Add semantic features to onchain model
    if 'onchain_semantic_cleaned' in feature_groups:
        onchain_feats = list(set(onchain_feats + feature_groups['onchain_semantic_cleaned']))
    
    print(f"\nPrice model: {len(price_feats)} features")
    print(f"Onchain model: {len(onchain_feats)} features (including semantics)")
    
    X_price = X[price_feats].fillna(method='ffill').fillna(0)
    X_onchain = X[onchain_feats].fillna(method='ffill').fillna(0)
    
    price_model = CalibratedClassifierCV(
        GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
        cv=3, method='sigmoid'
    ).fit(X_price, y)
    
    onchain_model = CalibratedClassifierCV(
        LogisticRegression(max_iter=1000),
        cv=3, method='sigmoid'
    ).fit(X_onchain, y)
    
    joblib.dump(price_model, 'data/price_model.pkl')
    joblib.dump(onchain_model, 'data/onchain_model.pkl')
    print("✅ Models saved")
    
    # 6️⃣ TEST DECISION
    engine = VetoDecisionEngine(price_model, onchain_model)
    decision = engine.predict(X_price.iloc[-1].values, X_onchain.iloc[-1].values)
    
    print(f"\n{'='*70}")
    print(f"🎯 LATEST DECISION: {decision}")
    print(f"{'='*70}")
    
    return engine, results

if __name__ == "__main__":
    engine, results = run_daily_pipeline()
    if engine:
        print("\n✅ Daily pipeline complete!")
    else:
        print("\n⚠️  Pipeline stopped at Step 1 audit")

ETH WHALE ML PIPELINE - WITH STEP 1 & 2 AUDITS

📂 LOADING DATA...
✅ dune_whales_cache.json current (2025-12-28)
✅ dune_intent_cache.json current (2025-12-28)
✅ BTC cache current (2025-12-28)
✅ ETH cache current (2025-12-28)

FEATURE ENGINEERING

🔧 Creating semantic whale behavior features...
✅ Created 8 semantic features: ['whale_accumulation', 'exchange_outflow_strength', 'whale_distribution', 'exchange_inflow_pressure', 'whale_panic_index', 'whale_conviction_index', 'whale_momentum_chase', 'whale_contrarian']

STEP 1 — FORMAL DIRECTIONAL TARGET AUDIT

1️⃣ GLOBAL CLASS BALANCE
----------------------------------------------------------------------
DOWN (0): 48.6%
UP   (1): 51.4%

Balance Status: ✅ EXCELLENT

2️⃣ PER-FOLD CLASS BALANCE (CRITICAL)
----------------------------------------------------------------------
Fold 1: DOWN=14, UP=16 ✅
Fold 2: DOWN=14, UP=16 ✅
Fold 3: DOWN=12, UP=18 ✅
Fold 4: DOWN=19, UP=11 ✅
Fold 5: DOWN=18, UP=12 ✅

✅ All folds have adequate samples

3️⃣ BASE-RAT

ModuleNotFoundError: No module named 'openpyxl'